In [ ]:
#part of the codes refer to https://github.com/AI4Finance-Foundation/FinRL
## install finrl library
!pip install git+https://github.com/AI4Finance-Foundation/FinRL.git

  Cloning https://github.com/AI4Finance-Foundation/FinRL.git to /tmp/pip-req-build-5j2f4izy
  Running command git clone --filter=blob:none --quiet https://github.com/AI4Finance-Foundation/FinRL.git /tmp/pip-req-build-5j2f4izy
  Resolved https://github.com/AI4Finance-Foundation/FinRL.git to commit 69776b349ee4e63efe3826f318aef8e5c5f59648
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/AI4Finance-Foundation/ElegantRL.git to /tmp/pip-install-ypxpetwc/elegantrl_21843ab61b894a3da1792710972d7fa8
  Running command git clone --filter=blob:none --quiet https://github.com/AI4Finance-Foundation/ElegantRL.git /tmp/pip-install-ypxpetwc/elegantrl_21843ab61b894a3da1792710972d7fa8
  Resolved https://github.com/AI4Finance-Foundation/ElegantRL.git to commit 5e828af1503098f4da046c0f12432dbd4ef8bd97
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.7/1

In [3]:
import pandas as pd
from stable_baselines3.common.logger import configure

from finrl.agents.stablebaselines3.models import DRLAgent
from finrl.config import INDICATORS, TRAINED_MODEL_DIR, RESULTS_DIR
from finrl.main import check_and_make_directories
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv

check_and_make_directories([TRAINED_MODEL_DIR])

In [2]:
!pip install pandas_market_calendars

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 198.4/198.4 kB 1.6 MB/s eta 0:00:00


In [4]:
train = pd.read_csv('train_data_final.csv')
train = train.set_index(train.columns[0])
train.index.names = ['']

In [5]:
import os
print(os.listdir())


['.config', 'train_data_final.csv', 'trade_data_final.csv', 'trained_models', 'trade_data_final_updated.csv', 'sample_data']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
#construct environment
stock_dimension = len(train.tic.unique())
state_space = 1 + 2*stock_dimension + len(INDICATORS)*stock_dimension
print(f"Stock Dimension: {stock_dimension}, State Space: {state_space}")

Stock Dimension: 29, State Space: 291


In [7]:
buy_cost_list = sell_cost_list = [0.001] * stock_dimension
num_stock_shares = [0] * stock_dimension

env_kwargs = {
    "hmax": 100,
    "initial_amount": 1000000,
    "num_stock_shares": num_stock_shares,
    "buy_cost_pct": buy_cost_list,
    "sell_cost_pct": sell_cost_list,
    "state_space": state_space,
    "stock_dim": stock_dimension,
    "tech_indicator_list": INDICATORS,
    "action_space": stock_dimension,
    "reward_scaling": 1e-4
}


e_train_gym = StockTradingEnv(df = train, **env_kwargs)

In [8]:
env_train, _ = e_train_gym.get_sb_env()
print(type(env_train))

<class 'stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv'>


In [9]:
agent = DRLAgent(env = env_train)

# Set the corresponding values to 'True' for the algorithms that you want to use
if_using_a2c = True
if_using_ddpg = True
if_using_ppo = True
if_using_td3 = True
if_using_sac = True

## Agent Training: 5 algorithms (A2C, DDPG, PPO, TD3, SAC)


### Agent 1: A2C


In [ ]:
# Tuning A2C 
from itertools import product

# search space
n_steps_list = [32, 64]
learning_rate_list = [1e-4, 5e-5]
ent_coef_list = [0.0, 0.005]
gamma_list = [0.90, 0.95]
timesteps = 100000  # fixed

#all combinations of hyperparameters
for n_steps, learning_rate, ent_coef, gamma in product(n_steps_list, learning_rate_list, ent_coef_list, gamma_list):
    model_kwargs = {
        "n_steps": n_steps,
        "learning_rate": learning_rate,
        "ent_coef": ent_coef,
        "gamma": gamma
    }

    print(f" Training A2C with n_steps={n_steps}, lr={learning_rate}, ent_coef={ent_coef}, gamma={gamma}, timesteps={timesteps}")
    log_dir = f"{RESULTS_DIR}/a2c_ns{n_steps}_lr{learning_rate}_ec{ent_coef}_gm{gamma}_ts{timesteps}"
    os.makedirs(log_dir, exist_ok=True)
    new_logger = configure(log_dir, ["stdout", "csv", "tensorboard"])

    agent = DRLAgent(env=env_train)
    model = agent.get_model("a2c", model_kwargs=model_kwargs)
    model.set_logger(new_logger)
    trained_model = agent.train_model(
        model=model,
        tb_log_name=f"a2c_ns{n_steps}_lr{learning_rate}_ec{ent_coef}_gm{gamma}_ts{timesteps}",
        total_timesteps=timesteps
    )
    model_filename = f"agent_a2c_ns{n_steps}_lr{learning_rate}_ec{ent_coef}_gm{gamma}_ts{timesteps}.zip"
    save_path = os.path.join(TRAINED_MODEL_DIR, model_filename)
    trained_model.save(save_path)
    print(f"Saved model to: {save_path}")



 Training A2C with n_steps=32, lr=0.0001, ent_coef=0.0, gamma=0.9, timesteps=100000
Logging to results/a2c_ns32_lr0.0001_ec0.0_gm0.9_ts100000
{'n_steps': 32, 'learning_rate': 0.0001, 'ent_coef': 0.0, 'gamma': 0.9}
Using cuda device


/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run A2C on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


--------------------------------------
| time/                 |            |
|    fps                | 114        |
|    iterations         | 100        |
|    time_elapsed       | 27         |
|    total_timesteps    | 3200       |
| train/                |            |
|    entropy_loss       | -41.1      |
|    explained_variance | -0.00116   |
|    learning_rate      | 0.0001     |
|    n_updates          | 99         |
|    policy_loss        | -109       |
|    reward             | -0.8612702 |
|    std                | 1          |
|    value_loss         | 13.6       |
--------------------------------------
--------------------------------------
| time/                 |            |
|    fps                | 114        |
|    iterations         | 200        |
|    time_elapsed       | 55         |
|    total_timesteps    | 6400       |
| train/                |            |
|    entropy_loss       | -41.1      |
|    explained_variance | -0.0708    |
|    learning_rate      |

/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run A2C on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


--------------------------------------
| time/                 |            |
|    fps                | 115        |
|    iterations         | 100        |
|    time_elapsed       | 27         |
|    total_timesteps    | 3200       |
| train/                |            |
|    entropy_loss       | -41.1      |
|    explained_variance | -0.108     |
|    learning_rate      | 0.0001     |
|    n_updates          | 99         |
|    policy_loss        | -114       |
|    reward             | -0.7336858 |
|    std                | 0.999      |
|    value_loss         | 13.8       |
--------------------------------------
--------------------------------------
| time/                 |            |
|    fps                | 115        |
|    iterations         | 200        |
|    time_elapsed       | 55         |
|    total_timesteps    | 6400       |
| train/                |            |
|    entropy_loss       | -41.1      |
|    explained_variance | -0.0745    |
|    learning_rate      |

...

--------------------------------------
| time/                 |            |
|    fps                | 116        |
|    iterations         | 100        |
|    time_elapsed       | 55         |
|    total_timesteps    | 6400       |
| train/                |            |
|    entropy_loss       | -41.2      |
|    explained_variance | 0.0177     |
|    learning_rate      | 5e-05      |
|    n_updates          | 99         |
|    policy_loss        | -4.85      |
|    reward             | -0.5790951 |
|    std                | 1          |
|    value_loss         | 2.18       |
--------------------------------------
day: 2892, episode: 470
begin_total_asset: 1000000.00
end_total_asset: 3310416.34
total_reward: 2310416.34
total_cost: 311673.96
total_trades: 80088
Sharpe: 0.650
---------------------------------------
| time/                 |             |
|    fps                | 116         |
|    iterations         | 200         |
|    time_elapsed       | 110         |
|    total_ti

/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run A2C on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


-------------------------------------
| time/                 |           |
|    fps                | 116       |
|    iterations         | 100       |
|    time_elapsed       | 54        |
|    total_timesteps    | 6400      |
| train/                |           |
|    entropy_loss       | -41.2     |
|    explained_variance | 0.106     |
|    learning_rate      | 5e-05     |
|    n_updates          | 99        |
|    policy_loss        | 38.6      |
|    reward             | -2.037848 |
|    std                | 1         |
|    value_loss         | 3.42      |
-------------------------------------
----------------------------------------
| time/                 |              |
|    fps                | 117          |
|    iterations         | 200          |
|    time_elapsed       | 109          |
|    total_timesteps    | 12800        |
| train/                |              |
|    entropy_loss       | -41.2        |
|    explained_variance | 0.0395       |
|    learning_rate     

/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run A2C on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


--------------------------------------
| time/                 |            |
|    fps                | 117        |
|    iterations         | 100        |
|    time_elapsed       | 54         |
|    total_timesteps    | 6400       |
| train/                |            |
|    entropy_loss       | -41.2      |
|    explained_variance | -0.0728    |
|    learning_rate      | 5e-05      |
|    n_updates          | 99         |
|    policy_loss        | 11.1       |
|    reward             | -1.2914317 |
|    std                | 1          |
|    value_loss         | 2.75       |
--------------------------------------
day: 2892, episode: 540
begin_total_asset: 1000000.00
end_total_asset: 3491805.34
total_reward: 2491805.34
total_cost: 324410.99
total_trades: 80858
Sharpe: 0.760
--------------------------------------
| time/                 |            |
|    fps                | 117        |
|    iterations         | 200        |
|    time_elapsed       | 109        |
|    total_timeste

In [10]:
#train = pd.read_csv('train_data.csv')
trade = pd.read_csv('trade_data_final_updated.csv')

# If you are not using the data generated from part 1 of this tutorial, make sure
# it has the columns and index in the form that could be make into the environment.
# Then you can comment and skip the following lines.
#train = train.set_index(train.columns[0])
#train.index.names = ['']
trade = trade.set_index(trade.columns[0])
trade.index.names = ['']

In [11]:
from stable_baselines3 import A2C, DDPG, PPO, SAC, TD3
#if_using_a2c = True
#trained_a2c = A2C.load("trained_models/agent_a2c") if if_using_a2c else None

In [12]:
stock_dimension = len(trade.tic.unique())
state_space = 1 + 2 * stock_dimension + len(INDICATORS) * stock_dimension
print(f"Stock Dimension: {stock_dimension}, State Space: {state_space}")

Stock Dimension: 29, State Space: 291


In [13]:
buy_cost_list = sell_cost_list = [0.001] * stock_dimension
num_stock_shares = [0] * stock_dimension

env_kwargs = {
    "hmax": 100,
    "initial_amount": 1000000,
    "num_stock_shares": num_stock_shares,
    "buy_cost_pct": buy_cost_list,
    "sell_cost_pct": sell_cost_list,
    "state_space": state_space,
    "stock_dim": stock_dimension,
    "tech_indicator_list": INDICATORS,
    "action_space": stock_dimension,
    "reward_scaling": 1e-4
}

In [14]:
print(trade.columns.tolist())
print(trade.head())


['date', 'tic', 'close', 'high', 'low', 'open', 'volume', 'day', 'macd', 'boll_ub', 'boll_lb', 'rsi_30', 'cci_30', 'dx_30', 'close_30_sma', 'close_60_sma', 'vix', 'turbulence']
         date   tic       close        high         low        open  \
                                                                      
0  2020-07-01  AAPL   88.601051   91.839996   90.977501   91.279999   
0  2020-07-01  AMGN  220.265930  256.230011  232.580002  235.520004   
0  2020-07-01   AXP   88.493973   96.959999   93.639999   95.250000   
0  2020-07-01    BA  180.320007  190.610001  180.039993  185.880005   
0  2020-07-01   CAT  114.372620  129.399994  125.879997  129.380005   

        volume  day      macd     boll_ub     boll_lb     rsi_30      cci_30  \
                                                                               
0  110737200.0  2.0  2.970893   91.355237   79.017396  62.807160  107.514852   
0    6575800.0  2.0  3.335248  213.151738  183.632585  61.279654  269.417140   
0    

In [15]:
e_trade_gym = StockTradingEnv(df = trade, turbulence_threshold = 70,risk_indicator_col='vix', **env_kwargs)

In [ ]:
df_account_value_a2c, df_actions_a2c = DRLAgent.DRL_prediction(
    model=trained_a2c,
    environment = e_trade_gym) if if_using_a2c else (None, None)

hit end!


In [ ]:
import pandas as pd
import numpy as np
from finrl.agents.stablebaselines3.models import DRLAgent

#backtesting on trade data, selecting models

trade = pd.read_csv('trade_data_final_updated.csv')
trade = trade.set_index(trade.columns[0])
trade.index.names = ['']

stock_dimension = len(trade.tic.unique())
state_space = 1 + 2 * stock_dimension + len(INDICATORS) * stock_dimension
buy_cost_list = sell_cost_list = [0.001] * stock_dimension
num_stock_shares = [0] * stock_dimension

env_kwargs = {
    "hmax": 100,
    "initial_amount": 1000000,
    "num_stock_shares": num_stock_shares,
    "buy_cost_pct": buy_cost_list,
    "sell_cost_pct": sell_cost_list,
    "state_space": state_space,
    "stock_dim": stock_dimension,
    "tech_indicator_list": INDICATORS,
    "action_space": stock_dimension,
    "reward_scaling": 1e-4
}

#evaluate model
result = pd.DataFrame()
TRAINED_MODEL_DIR = "trained_models"

for filename in os.listdir(TRAINED_MODEL_DIR):
    if filename.endswith(".zip") and filename.startswith("agent_a2c"):
        model_name = filename.replace(".zip", "")
        model_path = os.path.join(TRAINED_MODEL_DIR, filename)

        print(f"Evaluating model: {model_name}")
        trained_model = A2C.load(model_path)

        e_trade_gym = StockTradingEnv(
            df=trade,
            turbulence_threshold=70,
            risk_indicator_col='vix',
            **env_kwargs
        )

        df_account_value, _ = DRLAgent.DRL_prediction(
            model=trained_model,
            environment=e_trade_gym
        )

        df_account_value = df_account_value.set_index(df_account_value.columns[0])
        result[model_name] = df_account_value['account_value']

# Compute metrics

def performance_metrics(series: pd.Series, trading_days_per_year=252):
    series = series.dropna()
    v0 = series.iloc[0]
    vt = series.iloc[-1]
    daily_ret = series.pct_change().dropna()
    n = len(daily_ret)
    cum_ret = vt / v0 - 1
    ann_return = (vt / v0) ** (trading_days_per_year / n) - 1
    ann_vol = daily_ret.std() * np.sqrt(trading_days_per_year)
    sharpe = ann_return / ann_vol if ann_vol != 0 else np.nan
    running_max = series.cummax()
    drawdown = (series - running_max) / running_max
    max_dd = drawdown.min()
    return {
        "Initial": v0,
        "Final": vt,
        "Annualized Return": ann_return,
        "Annualized Std": ann_vol,
        "Sharpe Ratio": sharpe,
        "Max Drawdown": max_dd
    }

metrics = {col: performance_metrics(result[col]) for col in result.columns}
metrics_df = pd.DataFrame(metrics).T

metrics_df["Annualized Return"] = metrics_df["Annualized Return"].map("{:.2%}".format)
metrics_df["Annualized Std"]    = metrics_df["Annualized Std"].map("{:.2%}".format)
metrics_df["Sharpe Ratio"]      = metrics_df["Sharpe Ratio"].map("{:.2f}".format)
metrics_df["Max Drawdown"]      = metrics_df["Max Drawdown"].map("{:.2%}".format)

print(" Performance Summary:")
print(metrics_df)

result.to_csv("model_account_value_timeseries.csv")
metrics_df.to_csv("model_metrics_summary.csv")



Evaluating model: agent_a2c_ns64_lr0.0001_ec0.0_gm0.95_ts100000


/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run A2C on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


hit end!
Evaluating model: agent_a2c_ns32_lr5e-05_ec0.005_gm0.9_ts100000


/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run A2C on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


hit end!
Evaluating model: agent_a2c_ns64_lr5e-05_ec0.005_gm0.9_ts100000


...

hit end!
Evaluating model: agent_a2c_ns32_lr5e-05_ec0.0_gm0.9_ts100000


/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run A2C on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


hit end!
Evaluating model: agent_a2c_ns32_lr5e-05_ec0.005_gm0.95_ts100000


/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run A2C on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


hit end!
 Performance Summary:
                                                   Initial         Final  \
agent_a2c_ns64_lr0.0001_ec0.0_gm0.95_ts100000    1000000.0  2.082883e+06   
agent_a2c_ns32_lr5e-05_ec0.005_gm0.9_ts100000    1000000.0  1.510647e+06   
agent_a2c_ns64_lr5e-05_ec0.005_gm0.9_ts100000    1000000.0  1.867498e+06   
agent_a2c_ns32_lr0.0001_ec0.005_gm0.9_ts100000   1000000.0  1.825566e+06   
agent_a2c_ns64_lr0.0001_ec0.005_gm0.9_ts100000   1000000.0  1.790457e+06   
agent_a2c_ns32_lr0.0001_ec0.005_gm0.95_ts100000  1000000.0  1.668250e+06   
agent_a2c_ns64_lr5e-05_ec0.0_gm0.95_ts100000     1000000.0  1.879511e+06   
agent_a2c_ns64_lr5e-05_ec0.005_gm0.95_ts100000   1000000.0  1.659003e+06   
agent_a2c_ns64_lr0.0001_ec0.0_gm0.9_ts100000     1000000.0  1.516857e+06   
agent_a2c_ns32_lr0.0001_ec0.0_gm0.95_ts100000    1000000.0  1.342801e+06   
agent_a2c_ns32_lr0.0001_ec0.0_gm0.9_ts100000     1000000.0  1.833421e+06   
agent_a2c_ns64_lr0.0001_ec0.005_gm0.95_ts100000  1000000.

### Agent 2: DDPG

In [ ]:
#tuning DDPG
buffer_size_list = [50000, 100000]
learning_rate_list = [1e-3, 5e-4]
batch_size_list = [64, 128]
tau_list = [0.005, 0.01]
timesteps = 100000  

os.makedirs(TRAINED_MODEL_DIR, exist_ok=True)

for buffer_size, learning_rate, batch_size, tau in product(buffer_size_list, learning_rate_list, batch_size_list, tau_list):

    model_kwargs = {
        "buffer_size": buffer_size,
        "learning_rate": learning_rate,
        "batch_size": batch_size,
        "tau": tau,
        "gamma": 0.99
    }

    print(f"\nTraining DDPG with buffer={buffer_size}, lr={learning_rate}, batch={batch_size}, tau={tau}")

    log_name = f"ddpg_buf{buffer_size}_lr{learning_rate}_bs{batch_size}_tau{tau}_ts{timesteps}"
    tmp_path = os.path.join(RESULTS_DIR, log_name)
    os.makedirs(tmp_path, exist_ok=True)
    new_logger = configure(tmp_path, ["stdout", "csv", "tensorboard"])

    agent = DRLAgent(env=env_train)
    model = agent.get_model("ddpg", model_kwargs=model_kwargs)
    model.set_logger(new_logger)

    trained_model = agent.train_model(
        model=model,
        tb_log_name=log_name,
        total_timesteps=timesteps
    )

    save_path = os.path.join(TRAINED_MODEL_DIR, f"agent_{log_name}.zip")
    trained_model.save(save_path)
    print(f"Saved model to {save_path}")



Training DDPG with buffer=50000, lr=0.001, batch=64, tau=0.005
Logging to results/ddpg_buf50000_lr0.001_bs64_tau0.005_ts100000
{'buffer_size': 50000, 'learning_rate': 0.001, 'batch_size': 64, 'tau': 0.005, 'gamma': 0.99}
Using cuda device
----------------------------------
| time/              |           |
|    episodes        | 4         |
|    fps             | 72        |
|    time_elapsed    | 159       |
|    total_timesteps | 11572     |
| train/             |           |
|    actor_loss      | -10.6     |
|    critic_loss     | 47.5      |
|    learning_rate   | 0.001     |
|    n_updates       | 11471     |
|    reward          | 8.1817665 |
----------------------------------
day: 2892, episode: 580
begin_total_asset: 1000000.00
end_total_asset: 5860638.73
total_reward: 4860638.73
total_cost: 999.00
total_trades: 31812
Sharpe: 0.889
----------------------------------
| time/              |           |
|    episodes        | 8         |
|    fps             | 72        |
|    

In [ ]:
from stable_baselines3 import DDPG

result = pd.DataFrame()
model_files = [f for f in os.listdir() if f.endswith(".zip") and "ddpg" in f.lower()]

for filename in model_files:
    model_name = filename.replace(".zip", "")
    print(f"\nEvaluating model: {model_name}")
    trained_model = DDPG.load(filename)
    e_trade_gym = StockTradingEnv(
        df=trade,
        turbulence_threshold=70,
        risk_indicator_col='vix',
        **env_kwargs
    )

    df_account_value, _ = DRLAgent.DRL_prediction(
        model=trained_model,
        environment=e_trade_gym
    )

    df_account_value = df_account_value.set_index(df_account_value.columns[0])
    result[model_name] = df_account_value['account_value']


def performance_metrics(series: pd.Series, trading_days_per_year=252):
    series = series.dropna()
    v0 = series.iloc[0]
    vt = series.iloc[-1]
    daily_ret = series.pct_change().dropna()
    n = len(daily_ret)
    ann_return = (vt / v0) ** (trading_days_per_year / n) - 1
    ann_vol = daily_ret.std() * np.sqrt(trading_days_per_year)
    sharpe = ann_return / ann_vol if ann_vol != 0 else np.nan
    running_max = series.cummax()
    drawdown = (series - running_max) / running_max
    max_dd = drawdown.min()
    return {

        "Annualized Return": ann_return,
        "Annualized Std": ann_vol,
        "Sharpe Ratio": sharpe,
        "Max Drawdown": max_dd
    }

metrics = {col: performance_metrics(result[col]) for col in result.columns}
metrics_df = pd.DataFrame(metrics).T

metrics_df["Annualized Return"] = metrics_df["Annualized Return"].map("{:.2%}".format)
metrics_df["Annualized Std"] = metrics_df["Annualized Std"].map("{:.2%}".format)
metrics_df["Sharpe Ratio"] = metrics_df["Sharpe Ratio"].map("{:.2f}".format)
metrics_df["Max Drawdown"] = metrics_df["Max Drawdown"].map("{:.2%}".format)

# Display and save results
print("\nPerformance Summary:")
print(metrics_df)

result.to_csv("ddpg_model_account_value_timeseries.csv")
metrics_df.to_csv("ddpg_model_metrics_summary.csv")


Evaluating model: agent_ddpg_buf50000_lr0.0005_bs64_tau0.005_ts100000
hit end!

Evaluating model: agent_ddpg_buf50000_lr0.001_bs128_tau0.01_ts100000
hit end!

Evaluating model: agent_ddpg_buf50000_lr0.001_bs128_tau0.005_ts100000
hit end!

Evaluating model: agent_ddpg_buf50000_lr0.001_bs64_tau0.01_ts100000
hit end!

Evaluating model: agent_ddpg_buf50000_lr0.001_bs64_tau0.005_ts100000
hit end!

Evaluating model: agent_ddpg_buf100000_lr0.001_bs64_tau0.01_ts100000
hit end!

Evaluating model: agent_ddpg_buf50000_lr0.0005_bs128_tau0.01_ts100000
hit end!

Evaluating model: agent_ddpg_buf100000_lr0.001_bs128_tau0.005_ts100000
hit end!

Evaluating model: agent_ddpg_buf50000_lr0.0005_bs64_tau0.01_ts100000
hit end!

Evaluating model: agent_ddpg_buf100000_lr0.001_bs64_tau0.005_ts100000
hit end!

Evaluating model: agent_ddpg_buf50000_lr0.0005_bs128_tau0.005_ts100000
hit end!

Performance Summary:
                                                   Annualized Return  \
agent_ddpg_buf50000_lr0.0005_b

### Agent 3: PPO

In [ ]:
agent = DRLAgent(env = env_train)
PPO_PARAMS = {
    "n_steps": 2048,
    "ent_coef": 0.01,
    "learning_rate": 0.00025,
    "batch_size": 128,
}
model_ppo = agent.get_model("ppo",model_kwargs = PPO_PARAMS)

if if_using_ppo:
  # set up logger
  tmp_path = RESULTS_DIR + '/ppo'
  new_logger_ppo = configure(tmp_path, ["stdout", "csv", "tensorboard"])
  # Set new logger
  model_ppo.set_logger(new_logger_ppo)

{'n_steps': 2048, 'ent_coef': 0.01, 'learning_rate': 0.00025, 'batch_size': 128}
Using cuda device
Logging to results/ppo


/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


In [ ]:
trained_ppo = agent.train_model(model=model_ppo,
                             tb_log_name='ppo',
                             total_timesteps=200000) if if_using_ppo else None

----------------------------------
| time/              |           |
|    fps             | 116       |
|    iterations      | 1         |
|    time_elapsed    | 17        |
|    total_timesteps | 2048      |
| train/             |           |
|    reward          | 0.1628826 |
----------------------------------
----------------------------------------
| time/                   |            |
|    fps                  | 114        |
|    iterations           | 2          |
|    time_elapsed         | 35         |
|    total_timesteps      | 4096       |
| train/                  |            |
|    approx_kl            | 0.01784065 |
|    clip_fraction        | 0.211      |
|    clip_range           | 0.2        |
|    entropy_loss         | -41.2      |
|    explained_variance   | -0.0376    |
|    learning_rate        | 0.00025    |
|    loss                 | 5.92       |
|    n_updates            | 10         |
|    policy_gradient_loss | -0.0233    |
|    reward               | 0

In [ ]:
trained_ppo.save(TRAINED_MODEL_DIR + "/agent_ppo") if if_using_ppo else None

In [ ]:
df_account_value_ppo, df_actions_ppo = DRLAgent.DRL_prediction(
    model=trained_ppo,
    environment = e_trade_gym) if if_using_ppo else (None, None)

hit end!


### Agent 4: TD3

In [ ]:
agent = DRLAgent(env = env_train)
TD3_PARAMS = {"batch_size": 100,
              "buffer_size": 1000000,
              "learning_rate": 0.001}

model_td3 = agent.get_model("td3",model_kwargs = TD3_PARAMS)

if if_using_td3:
  # set up logger
  tmp_path = RESULTS_DIR + '/td3'
  new_logger_td3 = configure(tmp_path, ["stdout", "csv", "tensorboard"])
  # Set new logger
  model_td3.set_logger(new_logger_td3)

{'batch_size': 100, 'buffer_size': 1000000, 'learning_rate': 0.001}
Using cuda device
Logging to results/td3


In [ ]:
trained_td3 = agent.train_model(model=model_td3,
                             tb_log_name='td3',
                             total_timesteps=100000) if if_using_td3 else None

day: 2892, episode: 110
begin_total_asset: 1000000.00
end_total_asset: 3765104.45
total_reward: 2765104.45
total_cost: 999.00
total_trades: 40488
Sharpe: 0.706
---------------------------------
| time/              |          |
|    episodes        | 4        |
|    fps             | 77       |
|    time_elapsed    | 148      |
|    total_timesteps | 11572    |
| train/             |          |
|    actor_loss      | 176      |
|    critic_loss     | 44.7     |
|    learning_rate   | 0.001    |
|    n_updates       | 11471    |
|    reward          | 4.345844 |
---------------------------------
---------------------------------
| time/              |          |
|    episodes        | 8        |
|    fps             | 77       |
|    time_elapsed    | 299      |
|    total_timesteps | 23144    |
| train/             |          |
|    actor_loss      | 110      |
|    critic_loss     | 671      |
|    learning_rate   | 0.001    |
|    n_updates       | 23043    |
|    reward          | 4

In [ ]:
trained_td3.save(TRAINED_MODEL_DIR + "/agent_td3") if if_using_td3 else None

In [ ]:
df_account_value_td3, df_actions_td3 = DRLAgent.DRL_prediction(
    model=trained_td3,
    environment = e_trade_gym) if if_using_td3 else (None, None)

hit end!


### Agent 5: SAC

In [ ]:
agent = DRLAgent(env = env_train)
SAC_PARAMS = {
    "batch_size": 128,
    "buffer_size": 100000,
    "learning_rate": 0.0001,
    "learning_starts": 100,
    "ent_coef": "auto_0.1",
}

model_sac = agent.get_model("sac",model_kwargs = SAC_PARAMS)

if if_using_sac:
  # set up logger
  tmp_path = RESULTS_DIR + '/sac'
  new_logger_sac = configure(tmp_path, ["stdout", "csv", "tensorboard"])
  # Set new logger
  model_sac.set_logger(new_logger_sac)

{'batch_size': 128, 'buffer_size': 100000, 'learning_rate': 0.0001, 'learning_starts': 100, 'ent_coef': 'auto_0.1'}
Using cuda device
Logging to results/sac


In [ ]:
trained_sac = agent.train_model(model=model_sac,
                             tb_log_name='sac',
                             total_timesteps=100000) if if_using_sac else None

In [ ]:
trained_sac.save(TRAINED_MODEL_DIR + "/agent_sac") if if_using_sac else None

In [ ]:
import numpy as np

In [ ]:
df_account_value_sac, df_actions_sac = DRLAgent.DRL_prediction(
    model=trained_sac,
    environment = e_trade_gym) if if_using_sac else (None, None)

hit end!


In [ ]:
def process_df_for_mvo(df):
  return df.pivot(index="date", columns="tic", values="close")
# Codes in this section partially refer to Dr G A Vijayalakshmi Pai
# https://www.kaggle.com/code/vijipai/lesson-5-mean-variance-optimization-of-portfolios/notebook

def StockReturnsComputing(StockPrice, Rows, Columns):
  import numpy as np
  StockReturn = np.zeros([Rows-1, Columns])
  for j in range(Columns):        # j: Assets
    for i in range(Rows-1):     # i: Daily Prices
      StockReturn[i,j]=((StockPrice[i+1, j]-StockPrice[i,j])/StockPrice[i,j])* 100

  return StockReturn

In [ ]:
#mean variance optimization
StockData = process_df_for_mvo(train)
TradeData = process_df_for_mvo(trade)

TradeData.to_numpy()

In [ ]:
#compute asset returns
arStockPrices = np.asarray(StockData)
[Rows, Cols]=arStockPrices.shape
arReturns = StockReturnsComputing(arStockPrices, Rows, Cols)

#compute mean returns and variance covariance matrix of returns
meanReturns = np.mean(arReturns, axis = 0)
covReturns = np.cov(arReturns, rowvar=False)

#set precision for printing results
np.set_printoptions(precision=3, suppress = True)

#display mean returns and variance-covariance matrix of returns
print('Mean returns of assets in k-portfolio 1\n', meanReturns)
print('Variance-Covariance matrix of returns\n', covReturns)

In [ ]:
from pypfopt.efficient_frontier import EfficientFrontier

ef_mean = EfficientFrontier(meanReturns, covReturns, weight_bounds=(0, 0.5))
raw_weights_mean = ef_mean.max_sharpe()
cleaned_weights_mean = ef_mean.clean_weights()
mvo_weights = np.array([1000000 * cleaned_weights_mean[i] for i in range(len(cleaned_weights_mean))])
mvo_weights

In [ ]:
LastPrice = np.array([1/p for p in StockData.tail(1).to_numpy()[0]])
Initial_Portfolio = np.multiply(mvo_weights, LastPrice)
Initial_Portfolio

In [ ]:
Portfolio_Assets = TradeData @ Initial_Portfolio
MVO_result = pd.DataFrame(Portfolio_Assets, columns=["Mean Var"])
MVO_result

In [ ]:
TRAIN_START_DATE = '2009-01-01'
TRAIN_END_DATE = '2020-07-01'
TRADE_START_DATE = '2024-03-31'
TRADE_END_DATE = '2025-03-31'

In [ ]:
from finrl.meta.preprocessor.yahoodownloader import YahooDownloader

df_dji = YahooDownloader(
    start_date=TRADE_START_DATE, end_date=TRADE_END_DATE, ticker_list=["^dji"]
).fetch_data()

In [ ]:
df_dji = df_dji[["date", "close"]]
fst_day = df_dji["close"][0]
dji = pd.merge(
    df_dji["date"],
    df_dji["close"].div(fst_day).mul(1000000),
    how="outer",
    left_index=True,
    right_index=True,
).set_index("date")

In [ ]:
df_result_a2c = (
    df_account_value_a2c.set_index(df_account_value_a2c.columns[0])
    if if_using_a2c
    else None
)
df_result_ddpg = (
    df_account_value_ddpg.set_index(df_account_value_ddpg.columns[0])
    if if_using_ddpg
    else None
)
df_result_ppo = (
    df_account_value_ppo.set_index(df_account_value_ppo.columns[0])
    if if_using_ppo
    else None
)
df_result_td3 = (
    df_account_value_td3.set_index(df_account_value_td3.columns[0])
    if if_using_td3
    else None
)
df_result_sac = (
    df_account_value_sac.set_index(df_account_value_sac.columns[0])
    if if_using_sac
    else None
)

result = pd.DataFrame(
    {
        "a2c": df_result_a2c["account_value"] if if_using_a2c else None,
        "ddpg": df_result_ddpg["account_value"] if if_using_ddpg else None,
        "ppo": df_result_ppo["account_value"] if if_using_ppo else None,
        "td3": df_result_td3["account_value"] if if_using_td3 else None,
        "sac": df_result_sac["account_value"] if if_using_sac else None,
        "mvo": MVO_result["Mean Var"],
        "dji": dji["close"],
    }
)

In [ ]:
metrics = {col: performance_metrics(result[col]) for col in result.columns}
metrics_df = pd.DataFrame(metrics).T

metrics_df["Initial"] = metrics_df["Initial"].map("${:,.0f}".format)
metrics_df["Final"]   = metrics_df["Final"].map("${:,.0f}".format)
metrics_df["Annualized Return"] = metrics_df["Annualized Return"].map("{:.2%}".format)
metrics_df["Annualized Std"]    = metrics_df["Annualized Std"].map("{:.2%}".format)
metrics_df["Sharpe Ratio"]      = metrics_df["Sharpe Ratio"].map("{:.2f}".format)
metrics_df["Max Drawdown"]      = metrics_df["Max Drawdown"].map("{:.2%}".format)

print("\nPerformance Summary:")
print(metrics_df)

cumret = result.divide(result.iloc[0]).subtract(1.0)  # (V_t/V_0)-1


plt.figure(figsize=(12, 6))
for col in cumret.columns:
    plt.plot(cumret.index, cumret[col], label=col)
plt.legend()
plt.title("Cumulative Return Curve")
plt.ylabel("Cumulative Return")
plt.xlabel("Date")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
%matplotlib inline


In [ ]:
plt.figure(figsize=(12, 6))
for col in cumret.columns:
    plt.plot(cumret.index, cumret[col], label=col)
plt.legend()
plt.title("Cumulative Return Curve-Improved Model")
plt.ylabel("Cumulative Return")
plt.xlabel("Date")
plt.grid(True)
plt.tight_layout()
plt.show()
